# MG628 Week 8 Assignment Review: Inferential Statistics and Decision-Making

This is a worked review of the Week 8 assignment for all three scenarios. For every task you'll see:

1. **The task**: what the assignment asked for
2. **How to solve it**: the reasoning, the formula, the steps, and what to look for in the output
3. **The solution**: code you can run, with its output
4. **Model interpretation**: an example of a full-credit written answer, using the actual numbers

**Find your scenario:** Part A is Business (TechFlow premium onboarding), Part B is Healthcare Administration (telehealth recovery app), and Part C is Public Administration (Fast-Track vocational training). Every part follows the same seven tasks, so reading another track's part is good extra practice.

**Running it in Google Colab:** choose **Runtime → Run all**, or run cells from the top with **Shift + Enter**. The data loads directly from the MG628 GitHub. Use **File → Save a copy in Drive** if you want to experiment.

**How to use it:** compare it with your own notebook. Where your numbers differ, find the step where they diverge. Where your wording differs, check that your answer covers the same points: the number, the decision, the plain-English meaning, the practical size, and a limitation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import t as t_dist
import statsmodels.api as sm

BASE_URL = (
    "https://raw.githubusercontent.com/monroeu/MG628/refs/heads/main/week08/"
)
DATA = {
    "Business": BASE_URL + "business_conversion_data.csv",
    "Healthcare": BASE_URL + "healthcare_recovery.csv",
    "Public Administration": BASE_URL
    + "employment_program.csv",
}


def confidence_interval_95(data):
    n = len(data)
    mean = np.mean(data)
    std_err = stats.sem(data)
    margin = std_err * t_dist.ppf(0.975, n - 1)
    return mean, mean - margin, mean + margin


def cohens_d(treatment, control):
    pooled_sd = np.sqrt(
        (treatment.var(ddof=1) + control.var(ddof=1)) / 2
    )
    return (treatment.mean() - control.mean()) / pooled_sd

### Two general notes before you start

**1. Read direction from the means, not the sign of t.** The assignment templates call `stats.ttest_ind(control, treatment)`. Because Control is entered first, t comes out *negative* whenever Treatment scores higher. The sign only reflects the order you typed; the group means tell you which group is higher.

**2. Pooled vs. Welch's t-test.** The template uses the pooled t-test, which assumes both groups have equal variance. Levene's test checks that assumption; when its p-value is below 0.05, Welch's test (`equal_var=False`) is the safer choice. In all three datasets Levene's test is significant, but with 150 people in each group both tests give the same t-statistic to four decimals, so the conclusion doesn't change. A strong answer mentions the check.

---
# Part A: Business — TechFlow Premium Onboarding

**Scenario.** 150 customers received automated welcome emails (Control) and 150 received a 1-on-1 consultant video call (Treatment). Variables: `Customer_ID`, `Group`, `Satisfaction_Score` (scale 1–10), and `Monthly_Spend` (USD).

## Task 1: Load and explore the data (10 points)

**The task:** import the libraries, load the CSV, show the first 10 rows, run `info()` and `describe()`, and write 2–3 sentences describing each column and confirming data quality.

### How to solve it

- `head(10)` shows what one row looks like, so you can describe each column correctly.
- `info()` confirms the row count (300), the data types (the score and dollar columns should be numbers, not text), and the non-null counts. If every column shows 300 non-null values, there are no missing values.
- `describe()` gives the range of each numeric column. Check that scores stay between 1 and 10 and that dollar amounts are plausible (no negatives).
- `value_counts()` on `Group` confirms the 150/150 split.
- **Tip:** in a notebook, call `df.info()` on its own line. Wrapping it in `print()` also prints an extra `None`.

In [2]:
df = pd.read_csv(DATA["Business"])
display(df.head(10))
df.info()
display(df.describe())
print(df["Group"].value_counts())
print("Missing values per column:")
print(df.isna().sum())

HTTPError: HTTP Error 404: Not Found

### Model interpretation

The dataset has 300 rows and 4 columns: `Customer_ID` (a unique ID), `Group` (Control or Treatment, 150 each), `Satisfaction_Score` (a 1–10 rating), and `Monthly_Spend` (dollars). All columns are complete with no missing values, the score stays within 1–10, and the dollar values are plausible, so the data is ready for analysis.

## Task 2: Summary statistics by group (15 points)

**The task:** a table with the mean and standard deviation of `Satisfaction_Score` and `Monthly_Spend` for each group, plus a 3–4 sentence interpretation comparing the groups and stating the differences.

### How to solve it

1. `groupby('Group')` splits the data into Control and Treatment.
2. `.agg()` calculates the mean and standard deviation for each column within each group. That gives the 8 values the table needs.
3. **Calculate the differences yourself:** Treatment mean − Control mean, for both variables. Always state the order of subtraction.
4. **Interpret direction carefully.** For monthly spend, higher is better.

**Remember:** these are *descriptive* statistics. They describe these 300 customers but don't yet tell you whether the gap would hold for everyone. That's what Tasks 3 and 4 test.

In [ ]:
summary_stats = (
    df.groupby("Group")
    .agg(
        {
            "Satisfaction_Score": ["mean", "std"],
            "Monthly_Spend": ["mean", "std"],
        }
    )
    .round(2)
)
display(summary_stats)

means = df.groupby("Group")[
    ["Satisfaction_Score", "Monthly_Spend"]
].mean()
diff = means.loc["Treatment"] - means.loc["Control"]
print(
    "Satisfaction_Score difference (Treatment - Control): "
    f"{diff['Satisfaction_Score']:.2f} points"
)
print(
    "Monthly_Spend difference (Treatment - Control): "
    f"${diff['Monthly_Spend']:,.2f}"
)

### Model interpretation

Treatment customers rated their experience 7.82 on average versus 6.35 for Control, 1.48 points higher. They also spent $199.97 a month versus $146.51, about **$53.45 more per customer**. Treatment satisfaction is also slightly less spread out (SD 1.37 vs. 1.67). These descriptive results suggest premium onboarding raises both satisfaction and spending by more than its roughly $50 one-time cost, but the tests in Tasks 3–4 are needed before acting on it.

## Task 3: Two-sample t-test on Satisfaction_Score (15 points)

**The task:** run an independent two-sample t-test, report t (4 decimals) and the p-value, and explain the hypothesis decision, the business meaning, statistical significance, and the size of the difference.

### How to solve it

1. **Hypotheses.** H₀: the population mean satisfaction scores are equal. H₁: they differ (two-sided).
2. **Separate the two groups' scores**, then run `stats.ttest_ind(control, treatment)`.
3. **Decision rule:** if p < 0.05, reject H₀; otherwise fail to reject it. Never say "accept H₀."
4. **Check the variance assumption** with Levene's test, and compare with Welch's version.
5. **Report the size of the effect,** not only the p-value: the raw difference in points and Cohen's d (difference ÷ pooled SD; about 0.2 small, 0.5 medium, 0.8 large).
6. **Very small p-values** print as 0.0000; report them as p < 0.0001.

In [ ]:
control_s = df[df["Group"] == "Control"][
    "Satisfaction_Score"
]
treatment_s = df[df["Group"] == "Treatment"][
    "Satisfaction_Score"
]

t_stat, p_value = stats.ttest_ind(control_s, treatment_s)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4g}")
print(
    "Result:",
    (
        "SIGNIFICANT"
        if p_value < 0.05
        else "NOT SIGNIFICANT"
    ),
)

lev_p = stats.levene(control_s, treatment_s).pvalue
t_w, p_w = stats.ttest_ind(
    control_s, treatment_s, equal_var=False
)
print(
    f"\nLevene p = {lev_p:.4f}; Welch t = {t_w:.4f}, p = "
    f"{p_w:.4g}"
)
print(
    "Difference (Treatment - Control): "
    f"{treatment_s.mean() - control_s.mean():.2f} points"
)
print(
    f"Cohen's d: {cohens_d(treatment_s, control_s):.2f}"
)

### Model interpretation

The t-test gives **t = -8.3788, p < 0.0001**. Because p is far below 0.05, we **reject the null hypothesis** of equal average satisfaction. The t-statistic is negative only because Control was entered first; Treatment's mean is higher (7.82 vs. 6.35). A gap this large would be extremely unlikely if a 1-on-1 consultant video call had no effect on satisfaction, so it reflects a real difference, not chance. Practically, the gap is **1.48 points on the 1–10 satisfaction scale**, with Cohen's d = 0.97, a large effect. Levene's test (p = 0.0407) suggests unequal variances, but Welch's test gives the same conclusion.

## Task 4: 95% confidence intervals for Monthly_Spend (15 points)

**The task:** report the lower bound, mean, and upper bound for each group, then explain what each interval means, whether they overlap, and the practical difference in dollars.

### How to solve it

1. **Formula:** CI = mean ± t* × SE, where SE = SD ÷ √n and t* = `t_dist.ppf(0.975, n − 1)` (about 1.98 for n = 150).
2. **Run the function once per group.**
3. **Meaning:** "We are 95% confident the true average monthly spend for all customers like these lies between L and U." The interval describes the *average*, not individual customers.
4. **Overlap:** if the two intervals don't overlap, that is clear evidence of a real difference. Overlapping intervals are only a clue, not proof of no difference.
5. **For decisions,** a confidence interval for the *difference* is even more useful. It is included below as an extra.

In [ ]:
control_y = df[df["Group"] == "Control"]["Monthly_Spend"]
treatment_y = df[df["Group"] == "Treatment"][
    "Monthly_Spend"
]

mean_c, lower_c, upper_c = confidence_interval_95(
    control_y
)
mean_t, lower_t, upper_t = confidence_interval_95(
    treatment_y
)
print(
    f"Control:   mean ${mean_c:,.2f}, 95% CI "
    f"[${lower_c:,.2f}, ${upper_c:,.2f}]"
)
print(
    f"Treatment: mean ${mean_t:,.2f}, 95% CI "
    f"[${lower_t:,.2f}, ${upper_t:,.2f}]"
)
print(
    "Intervals overlap?",
    not (upper_c < lower_t or upper_t < lower_c),
)

# Extra: 95% CI for the difference (Welch)
diff = mean_t - mean_c
se_diff = np.sqrt(
    control_y.var(ddof=1) / len(control_y)
    + treatment_y.var(ddof=1) / len(treatment_y)
)
print(
    f"\nDifference (Treatment - Control): ${diff:,.2f}, "
    f"approx. 95% CI [${diff - 1.97 * se_diff:,.2f}, "
    f"${diff + 1.97 * se_diff:,.2f}]"
)

### Model interpretation

We are 95% confident that the true average spend is between **$141.06 and $151.97** for Control customers and between **$194.93 and $205.01** for Treatment customers. The intervals **do not overlap**, so the difference is very unlikely to be sampling noise. The average difference is **$53.45 per customer** more for Treatment, and the 95% CI for the difference runs from about $46.06 to $60.85. Even the low end exceeds the $50 program cost.

## Task 5: Regression analysis (20 points)

**The task:** regress `Monthly_Spend` on `Satisfaction_Score` and `Group_Treatment` (1 = Treatment, 0 = Control); report the full summary, R², both coefficients and p-values, and interpret them.

### How to solve it

1. **Create the dummy variable:** `Group_Treatment = 1` for Treatment, `0` for Control. Control becomes the reference group.
2. **Add a constant** (`sm.add_constant`) so the model has an intercept, then fit `sm.OLS(y, X)`.
3. **Read the output in this order:** R², then each coefficient (sign and size), then each p-value, then what the group effect means *after controlling for the score*.
   - **Satisfaction_Score coefficient:** the change in monthly spend for a 1-point higher score, *holding group constant*.
   - **Group_Treatment coefficient:** the difference between Treatment and Control *for customers with the same score*.
   - **R²:** the share of the variation in monthly spend the model explains (not "accuracy").
4. **Signs matter:** here a positive coefficient is good news (higher monthly spend).
5. **Why both effects matter:** if the group effect stays significant after controlling for the score, the program adds value *beyond* making people happier.

In [ ]:
df["Group_Treatment"] = (
    df["Group"] == "Treatment"
).astype(int)
X = sm.add_constant(
    df[["Satisfaction_Score", "Group_Treatment"]]
)
y = df["Monthly_Spend"]
model = sm.OLS(y, X).fit()
print(model.summary())

print(f"\nR-squared = {model.rsquared:.4f}")
for v in ["Satisfaction_Score", "Group_Treatment"]:
    print(
        f"{v}: beta = {model.params[v]:,.2f}, p = "
        f"{model.pvalues[v]:.4g}"
    )

### Model interpretation

The model explains **67.6%** of the variation in monthly spend (R² = 0.6756), a strong fit for behavioral data. **Satisfaction_Score:** β = **+$14.37** (p < 0.0001): each 1-point higher satisfaction is associated with $14.37 higher monthly spend, holding group constant. **Group_Treatment:** β = **+$32.21** (p < 0.0001): customers receiving a 1-on-1 consultant video call spend about $32.21 more than Control customers with the same satisfaction. Both predictors are significant on their own, so the program increases monthly spend through two paths: by raising satisfaction (1.48 points × $14.37 ≈ $21.24) and through a direct effect of about $32.21. Together these account for the $53.45 total difference.

## Task 6: Visualizations (15 points)

**The task:** an overlapping histogram and side-by-side box plots of `Satisfaction_Score`, each with labels, a title, and a legend where needed, plus 2–3 sentences interpreting each.

### How to solve it

- **Histogram:** plot both groups on the same axes with transparency (`alpha=0.6`) so the overlap shows. Use the same bins for both (10 bins across 1–10). Look at *where each group's bars pile up* and *which group has more high and low scores*.
- **Box plot:** the box spans Q1 to Q3 (the middle half), the line is the median, the whiskers reach the furthest point within 1.5 × IQR, and dots are outliers. Compare medians, box widths (consistency), and outliers.
- **Template fix:** in the assignment's code, `df.boxplot(by='Group')` opens a new figure and leaves the second subplot empty. Pass `ax=` to keep both charts side by side, as below.
- **Connect it to the decision:** say what the pattern means for the program, not only what the chart shows.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7, 9))
bins = np.linspace(1, 10, 11)
ax[0].hist(
    control_s,
    bins=bins,
    alpha=0.6,
    label="Control",
    color="#3D5A80",
    edgecolor="white",
)
ax[0].hist(
    treatment_s,
    bins=bins,
    alpha=0.6,
    label="Treatment",
    color="#E9A23B",
    edgecolor="white",
)
ax[0].set_xlabel("Satisfaction Score")
ax[0].set_ylabel("Frequency")
ax[0].set_title("Distribution of Satisfaction Score")
ax[0].legend()

df.boxplot(
    column="Satisfaction_Score", by="Group", ax=ax[1]
)
ax[1].set_xlabel("Group")
ax[1].set_ylabel("Satisfaction Score")
ax[1].set_title("Satisfaction Score by Group")
plt.suptitle("")
plt.tight_layout()
plt.show()

for g, sc in [
    ("Control", control_s),
    ("Treatment", treatment_s),
]:
    q1, q3 = sc.quantile([0.25, 0.75])
    print(
        f"{g}: median {sc.median():.2f}, IQR "
        f"{q3 - q1:.2f}, share scoring 8+: "
        f"{(sc >= 8).mean():.1%}"
    )

### Model interpretation

**Histogram.** Treatment scores are concentrated toward the top of the scale: 51% of Treatment customers scored 8 or higher, versus 17% of Control. Control scores are more spread out, with more low scores (22% scored 5 or below, versus 2.7% for Treatment). The two distributions overlap, so not every Treatment customer was happier than every Control customer, but the whole Treatment distribution has shifted to the right.

**Box plot.** Treatment's median (8.00) is well above Control's (6.40). Treatment's box is narrower (IQR 2.00 vs. 2.10), so its scores are more consistent and predictable. Control has one low outlier at 1.8; Treatment has one low outlier at 2.9, likely individuals who didn't engage with the program or had an unusually poor experience. These are worth following up rather than deleting. Together, the charts support the t-test: Treatment customers are consistently more satisfied.

## Task 7: Business recommendation memo (10 points)

**The task:** a 1–2 page memo in Markdown cells with a clear YES or NO, key findings with specific numbers, a cost-benefit analysis, a recommendation, and limitations.

### How to solve it

1. **Lead with the decision** in one or two sentences, with the main reason.
2. **Pull the key numbers** from Tasks 2–5. Don't introduce new ones.
3. **Do the cost-benefit math.** Cost: $50 per customer, one-time. Benefit: the Task 2 spending difference, per month. Break-even: months = cost ÷ monthly gain. Then test the **low end** of the confidence interval for the difference. If the decision still holds there, it's robust.
4. **Name the assumptions:** spending persists beyond month two, the pilot customers resemble future ones, and consultant capacity scales.

In [ ]:
cost = 50
gain = treatment_y.mean() - control_y.mean()
low = 46.06  # low end of the 95% CI for the difference (Task 4)
print(
    f"Monthly gain per customer: ${gain:,.2f} (low end "
    f"${low:,.2f})"
)
print(
    f"Months to recover $50: {cost / gain:.2f} (low end: "
    f"{cost / low:.2f})"
)
new_customers = 1000
print(
    f"First-year net impact for {new_customers:,} "
    "customers, if the gain lasts 12 months: "
    f"${new_customers * (12 * gain - cost):,.0f}"
)

### Model memo

**TO:** Marketing Director, TechFlow  **FROM:** Business Analyst  **DATE:** [date]  **SUBJECT:** Expanding Premium Onboarding

**Executive summary.** **YES. Expand Premium Onboarding to all new customers.** The $50 consultant call pays for itself within about a month through higher spending, and it also raises satisfaction.

**Key findings.**
- **Satisfaction:** Treatment customers rated 7.82 vs. 6.35 (t = -8.3788, p < 0.0001), a statistically significant and large difference (d = 0.97).
- **Spending:** $199.97 vs. $146.51 per month, a difference of **$53.45**.
- **Confidence intervals:** Control [$141.06, $151.97] and Treatment [$194.93, $205.01] do not overlap, so the spending gap is reliable.
- **Regression:** satisfaction β = $14.37 per point; Group_Treatment β = $32.21; R² = 0.68. Both are significant, so onboarding raises spending directly and through higher satisfaction.

**Cost-benefit analysis.** We spend $50 once and gain about $53.45 per customer per month, so the cost is recovered in **0.94 months**. Even at the low end of the 95% interval ($46.06 a month) it is recovered in about 1.1 months. For 1,000 new customers a year, the first-year net gain is roughly **$591,454.40**, if the higher spending lasts.

**Recommendation.** Roll out Premium Onboarding to all new customers, phased over one or two quarters so consultant hiring keeps pace. The evidence is strong on both satisfaction and revenue, and the payback is fast. Track month-3 to month-12 spending and churn to confirm that the gain persists.

**Considerations and limitations.** Results cover only the first two months of spending; we assume the gain persists and that pilot customers resemble future sign-ups. Consultant quality may slip at scale, and the cost may rise above $50 if hiring is competitive. Additional data on retention, customer lifetime value, and churn would strengthen the case.

---
# Part B: Healthcare Administration — Telehealth vs. In-Person Post-Operative Recovery

**Scenario.** 150 patients received in-person follow-up visits (Control) and 150 received the telehealth monitoring app (Treatment). Variables: `Patient_ID`, `Group`, `Satisfaction_Score` (scale 1–10), and `Recovery_Cost` (USD).

## Task 1: Load and explore the data (10 points)

**The task:** import the libraries, load the CSV, show the first 10 rows, run `info()` and `describe()`, and write 2–3 sentences describing each column and confirming data quality.

### How to solve it

- `head(10)` shows what one row looks like, so you can describe each column correctly.
- `info()` confirms the row count (300), the data types (the score and dollar columns should be numbers, not text), and the non-null counts. If every column shows 300 non-null values, there are no missing values.
- `describe()` gives the range of each numeric column. Check that scores stay between 1 and 10 and that dollar amounts are plausible (no negatives).
- `value_counts()` on `Group` confirms the 150/150 split.
- **Tip:** in a notebook, call `df.info()` on its own line. Wrapping it in `print()` also prints an extra `None`.

In [ ]:
df = pd.read_csv(DATA["Healthcare"])
display(df.head(10))
df.info()
display(df.describe())
print(df["Group"].value_counts())
print("Missing values per column:")
print(df.isna().sum())

### Model interpretation

The dataset has 300 rows and 4 columns: `Patient_ID` (a unique ID), `Group` (Control or Treatment, 150 each), `Satisfaction_Score` (a 1–10 rating), and `Recovery_Cost` (dollars). All columns are complete with no missing values, the score stays within 1–10, and the dollar values are plausible, so the data is ready for analysis.

## Task 2: Summary statistics by group (15 points)

**The task:** a table with the mean and standard deviation of `Satisfaction_Score` and `Recovery_Cost` for each group, plus a 3–4 sentence interpretation comparing the groups and stating the differences.

### How to solve it

1. `groupby('Group')` splits the data into Control and Treatment.
2. `.agg()` calculates the mean and standard deviation for each column within each group. That gives the 8 values the table needs.
3. **Calculate the differences yourself:** Treatment mean − Control mean, for both variables. Always state the order of subtraction.
4. **Interpret direction carefully.** For recovery cost, lower is better (lower costs are the goal).

**Remember:** these are *descriptive* statistics. They describe these 300 patients but don't yet tell you whether the gap would hold for everyone. That's what Tasks 3 and 4 test.

In [ ]:
summary_stats = (
    df.groupby("Group")
    .agg(
        {
            "Satisfaction_Score": ["mean", "std"],
            "Recovery_Cost": ["mean", "std"],
        }
    )
    .round(2)
)
display(summary_stats)

means = df.groupby("Group")[
    ["Satisfaction_Score", "Recovery_Cost"]
].mean()
diff = means.loc["Treatment"] - means.loc["Control"]
print(
    "Satisfaction_Score difference (Treatment - Control): "
    f"{diff['Satisfaction_Score']:.2f} points"
)
print(
    "Recovery_Cost difference (Treatment - Control): "
    f"${diff['Recovery_Cost']:,.2f}"
)

### Model interpretation

Telehealth patients rated satisfaction 8.20 on average versus 7.26 for in-person patients, 0.94 points higher. Their average recovery cost was $756.29 versus $1,112.86, a **saving of $356.57 per patient**. Telehealth costs were also less variable (SD $230.50 vs. $272.86). The initial picture is that the app improves the patient experience and reduces follow-up and complication costs, which the tests in Tasks 3–4 will check.

## Task 3: Two-sample t-test on Satisfaction_Score (15 points)

**The task:** run an independent two-sample t-test, report t (4 decimals) and the p-value, and explain the hypothesis decision, the practical meaning, statistical significance, and the size of the difference.

### How to solve it

1. **Hypotheses.** H₀: the population mean satisfaction scores are equal. H₁: they differ (two-sided).
2. **Separate the two groups' scores**, then run `stats.ttest_ind(control, treatment)`.
3. **Decision rule:** if p < 0.05, reject H₀; otherwise fail to reject it. Never say "accept H₀."
4. **Check the variance assumption** with Levene's test, and compare with Welch's version.
5. **Report the size of the effect,** not only the p-value: the raw difference in points and Cohen's d (difference ÷ pooled SD; about 0.2 small, 0.5 medium, 0.8 large).
6. **Very small p-values** print as 0.0000; report them as p < 0.0001.

In [ ]:
control_s = df[df["Group"] == "Control"][
    "Satisfaction_Score"
]
treatment_s = df[df["Group"] == "Treatment"][
    "Satisfaction_Score"
]

t_stat, p_value = stats.ttest_ind(control_s, treatment_s)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4g}")
print(
    "Result:",
    (
        "SIGNIFICANT"
        if p_value < 0.05
        else "NOT SIGNIFICANT"
    ),
)

lev_p = stats.levene(control_s, treatment_s).pvalue
t_w, p_w = stats.ttest_ind(
    control_s, treatment_s, equal_var=False
)
print(
    f"\nLevene p = {lev_p:.4f}; Welch t = {t_w:.4f}, p = "
    f"{p_w:.4g}"
)
print(
    "Difference (Treatment - Control): "
    f"{treatment_s.mean() - control_s.mean():.2f} points"
)
print(
    f"Cohen's d: {cohens_d(treatment_s, control_s):.2f}"
)

### Model interpretation

The t-test gives **t = -5.6830, p < 0.0001**. Because p is far below 0.05, we **reject the null hypothesis** of equal average satisfaction. The t-statistic is negative only because Control was entered first; Treatment's mean is higher (8.20 vs. 7.26). A gap this large would be extremely unlikely if the telehealth monitoring app had no effect on satisfaction, so it reflects a real difference, not chance. Practically, the gap is **0.94 points on the 1–10 satisfaction scale**, with Cohen's d = 0.66, a medium-to-large effect. Levene's test (p = 0.0000) suggests unequal variances, but Welch's test gives the same conclusion.

## Task 4: 95% confidence intervals for Recovery_Cost (15 points)

**The task:** report the lower bound, mean, and upper bound for each group, then explain what each interval means, whether they overlap, and the practical difference in dollars. (The instructions for this track say Monthly_Spend in places; for healthcare, use Recovery_Cost.)

### How to solve it

1. **Formula:** CI = mean ± t* × SE, where SE = SD ÷ √n and t* = `t_dist.ppf(0.975, n − 1)` (about 1.98 for n = 150).
2. **Run the function once per group.**
3. **Meaning:** "We are 95% confident the true average recovery cost for all patients like these lies between L and U." The interval describes the *average*, not individual patients.
4. **Overlap:** if the two intervals don't overlap, that is clear evidence of a real difference. Overlapping intervals are only a clue, not proof of no difference.
5. **For decisions,** a confidence interval for the *difference* is even more useful. It is included below as an extra.

In [ ]:
control_y = df[df["Group"] == "Control"]["Recovery_Cost"]
treatment_y = df[df["Group"] == "Treatment"][
    "Recovery_Cost"
]

mean_c, lower_c, upper_c = confidence_interval_95(
    control_y
)
mean_t, lower_t, upper_t = confidence_interval_95(
    treatment_y
)
print(
    f"Control:   mean ${mean_c:,.2f}, 95% CI "
    f"[${lower_c:,.2f}, ${upper_c:,.2f}]"
)
print(
    f"Treatment: mean ${mean_t:,.2f}, 95% CI "
    f"[${lower_t:,.2f}, ${upper_t:,.2f}]"
)
print(
    "Intervals overlap?",
    not (upper_c < lower_t or upper_t < lower_c),
)

# Extra: 95% CI for the difference (Welch)
diff = mean_t - mean_c
se_diff = np.sqrt(
    control_y.var(ddof=1) / len(control_y)
    + treatment_y.var(ddof=1) / len(treatment_y)
)
print(
    f"\nDifference (Treatment - Control): ${diff:,.2f}, "
    f"approx. 95% CI [${diff - 1.97 * se_diff:,.2f}, "
    f"${diff + 1.97 * se_diff:,.2f}]"
)

### Model interpretation

We are 95% confident that the true average recovery cost is between **$1,068.84 and $1,156.88** for Control patients and between **$719.10 and $793.48** for Treatment patients. The intervals **do not overlap**, so the difference is very unlikely to be sampling noise. The average difference is **$356.57 per patient** saved with telehealth, and the 95% CI for the difference runs from about $299.17 to $413.96. Even the low end is a substantial saving per patient, which is useful for budgeting.

## Task 5: Regression analysis (20 points)

**The task:** regress `Recovery_Cost` on `Satisfaction_Score` and `Group_Treatment` (1 = Treatment, 0 = Control); report the full summary, R², both coefficients and p-values, and interpret them. (The instructions say Monthly_Spend as the dependent variable in places; for healthcare, use Recovery_Cost, as the code template does.)

### How to solve it

1. **Create the dummy variable:** `Group_Treatment = 1` for Treatment, `0` for Control. Control becomes the reference group.
2. **Add a constant** (`sm.add_constant`) so the model has an intercept, then fit `sm.OLS(y, X)`.
3. **Read the output in this order:** R², then each coefficient (sign and size), then each p-value, then what the group effect means *after controlling for the score*.
   - **Satisfaction_Score coefficient:** the change in recovery cost for a 1-point higher score, *holding group constant*.
   - **Group_Treatment coefficient:** the difference between Treatment and Control *for patients with the same score*.
   - **R²:** the share of the variation in recovery cost the model explains (not "accuracy").
4. **Signs matter:** for costs, a negative coefficient is good news (it lowers cost).
5. **Why both effects matter:** if the group effect stays significant after controlling for the score, the program adds value *beyond* making people happier.

In [ ]:
df["Group_Treatment"] = (
    df["Group"] == "Treatment"
).astype(int)
X = sm.add_constant(
    df[["Satisfaction_Score", "Group_Treatment"]]
)
y = df["Recovery_Cost"]
model = sm.OLS(y, X).fit()
print(model.summary())

print(f"\nR-squared = {model.rsquared:.4f}")
for v in ["Satisfaction_Score", "Group_Treatment"]:
    print(
        f"{v}: beta = {model.params[v]:,.2f}, p = "
        f"{model.pvalues[v]:.4g}"
    )

### Model interpretation

The model explains **69.4%** of the variation in recovery cost (R² = 0.6944), a strong fit for behavioral data. **Satisfaction_Score:** β = **−$129.24** (p < 0.0001): each 1-point higher satisfaction is associated with $129.24 lower recovery cost, holding group constant. **Group_Treatment:** β = **−$234.66** (p < 0.0001): patients receiving the telehealth monitoring app cost about $234.66 less than Control patients with the same satisfaction. Both predictors are significant on their own, so the program reduces recovery cost through two paths: by raising satisfaction (0.94 points × $129.24 ≈ $121.91) and through a direct effect of about $234.66. Together these account for the $356.57 total difference.

## Task 6: Visualizations (15 points)

**The task:** an overlapping histogram and side-by-side box plots of `Satisfaction_Score`, each with labels, a title, and a legend where needed, plus 2–3 sentences interpreting each.

### How to solve it

- **Histogram:** plot both groups on the same axes with transparency (`alpha=0.6`) so the overlap shows. Use the same bins for both (10 bins across 1–10). Look at *where each group's bars pile up* and *which group has more high and low scores*.
- **Box plot:** the box spans Q1 to Q3 (the middle half), the line is the median, the whiskers reach the furthest point within 1.5 × IQR, and dots are outliers. Compare medians, box widths (consistency), and outliers.
- **Template fix:** in the assignment's code, `df.boxplot(by='Group')` opens a new figure and leaves the second subplot empty. Pass `ax=` to keep both charts side by side, as below.
- **Connect it to the decision:** say what the pattern means for the program, not only what the chart shows.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7, 9))
bins = np.linspace(1, 10, 11)
ax[0].hist(
    control_s,
    bins=bins,
    alpha=0.6,
    label="Control",
    color="#3D5A80",
    edgecolor="white",
)
ax[0].hist(
    treatment_s,
    bins=bins,
    alpha=0.6,
    label="Treatment",
    color="#E9A23B",
    edgecolor="white",
)
ax[0].set_xlabel("Satisfaction Score")
ax[0].set_ylabel("Frequency")
ax[0].set_title("Distribution of Satisfaction Score")
ax[0].legend()

df.boxplot(
    column="Satisfaction_Score", by="Group", ax=ax[1]
)
ax[1].set_xlabel("Group")
ax[1].set_ylabel("Satisfaction Score")
ax[1].set_title("Satisfaction Score by Group")
plt.suptitle("")
plt.tight_layout()
plt.show()

for g, sc in [
    ("Control", control_s),
    ("Treatment", treatment_s),
]:
    q1, q3 = sc.quantile([0.25, 0.75])
    print(
        f"{g}: median {sc.median():.2f}, IQR "
        f"{q3 - q1:.2f}, share scoring 8+: "
        f"{(sc >= 8).mean():.1%}"
    )

### Model interpretation

**Histogram.** Treatment scores are concentrated toward the top of the scale: 62% of Treatment patients scored 8 or higher, versus 35% of Control. Control scores are more spread out, with more low scores (9% scored 5 or below, versus 0.7% for Treatment). The two distributions overlap, so not every Treatment patient was happier than every Control patient, but the whole Treatment distribution has shifted to the right.

**Box plot.** Treatment's median (8.30) is well above Control's (7.30). Treatment's box is narrower (IQR 1.40 vs. 2.57), so its scores are more consistent and predictable. Control has no outliers; Treatment has low outliers at 4.2 and 5.1, likely individuals who didn't engage with the program or had an unusually poor experience. These are worth following up rather than deleting. Together, the charts support the t-test: Treatment patients are consistently more satisfied.

## Task 7: Recommendation recommendation memo (10 points)

**The task:** a 1–2 page memo in Markdown cells with a clear YES or NO, key findings with specific numbers, a cost-benefit analysis, a recommendation, and limitations.

### How to solve it

1. **Lead with the decision** and the main reason.
2. **Pull the key numbers** from Tasks 2–5.
3. **Cost-benefit math.** The assignment asks *you* to estimate the implementation costs. State them clearly as assumptions; the grader looks for reasonable, explained estimates, not one "right" number. Then: annual savings = savings per patient × patients per year; break-even patients = implementation cost ÷ savings per patient.
4. **Weigh patient experience and safety** alongside the dollars.
5. **Note:** the assignment's "$50 per customer" line belongs to the business scenario; healthcare uses your own implementation estimates.

In [ ]:
saving = (
    control_y.mean() - treatment_y.mean()
)  # per patient
# Illustrative assumptions: replace with your own, and explain them
licensing, training, it_support = 60_000, 15_000, 25_000
implementation = licensing + training + it_support
patients_per_year = 2_000
print(f"Savings per patient: ${saving:,.2f}")
print(
    "Year-1 implementation cost (assumed): "
    f"${implementation:,.0f}"
)
print(
    f"Annual savings for {patients_per_year:,} patients: "
    f"${saving * patients_per_year:,.0f}"
)
print(
    "Net year-1 benefit: "
    f"${saving * patients_per_year - implementation:,.0f}"
)
print(
    f"Break-even: {implementation / saving:,.0f} patients"
)

### Model memo

**TO:** Chief Operating Officer  **FROM:** Hospital Administrator  **DATE:** [date]  **SUBJECT:** Expanding the Telehealth Monitoring App

**Executive summary.** **YES. Expand the telehealth app to all eligible minor-surgery patients.** It saves about $356.57 per patient while raising satisfaction, and it covers its implementation cost after roughly 280 patients.

**Key findings.**
- **Satisfaction:** 8.20 vs. 7.26 (t = -5.6830, p < 0.0001), statistically significant, with a medium-to-large effect (d = 0.66).
- **Cost:** $756.29 vs. $1,112.86 per patient, a saving of **$356.57**.
- **Confidence intervals:** Control [$1,068.84, $1,156.88] and Treatment [$719.10, $793.48] do not overlap; the saving is reliable.
- **Regression:** satisfaction β = −$129.24 per point; Group_Treatment β = −$234.66; R² = 0.69. Both reduce costs independently.

**Cost-benefit analysis (illustrative assumptions).** Implementation: licensing $60,000, staff training $15,000, IT support $25,000, so about **$100,000** in year one. At 2,000 eligible surgical patients a year, savings are about $713,139.73, a net benefit of about **$613,139.73**. Break-even: $100,000 ÷ $356.57 ≈ **280 patients**.

**Recommendation.** Expand the app hospital-wide for the minor procedures studied, with a nurse-monitoring protocol and an easy route back to in-person care. Monitor readmissions, ER visits, and complication rates quarterly to confirm the savings and safety hold.

**Considerations and limitations.** The pilot covered minor surgeries; results may not apply to complex procedures or to patients without smartphones or digital skills, which raises equity concerns. Implementation costs are estimates. Additional data on readmissions, long-term outcomes, and nurse workload would strengthen the case.

---
# Part C: Public Administration — Fast-Track Vocational Training

**Scenario.** 150 participants received standard job-search assistance (Control) and 150 received the 12-week Fast-Track training program (Treatment). Variables: `Participant_ID`, `Group`, `Confidence_Score` (scale 1–10), and `Monthly_Income` (USD).

## Task 1: Load and explore the data (10 points)

**The task:** import the libraries, load the CSV, show the first 10 rows, run `info()` and `describe()`, and write 2–3 sentences describing each column and confirming data quality.

### How to solve it

- `head(10)` shows what one row looks like, so you can describe each column correctly.
- `info()` confirms the row count (300), the data types (the score and dollar columns should be numbers, not text), and the non-null counts. If every column shows 300 non-null values, there are no missing values.
- `describe()` gives the range of each numeric column. Check that scores stay between 1 and 10 and that dollar amounts are plausible (no negatives).
- `value_counts()` on `Group` confirms the 150/150 split.
- **Tip:** in a notebook, call `df.info()` on its own line. Wrapping it in `print()` also prints an extra `None`.

In [ ]:
df = pd.read_csv(DATA["Public Administration"])
display(df.head(10))
df.info()
display(df.describe())
print(df["Group"].value_counts())
print("Missing values per column:")
print(df.isna().sum())

### Model interpretation

The dataset has 300 rows and 4 columns: `Participant_ID` (a unique ID), `Group` (Control or Treatment, 150 each), `Confidence_Score` (a 1–10 rating), and `Monthly_Income` (dollars). All columns are complete with no missing values, the score stays within 1–10, and the dollar values are plausible, so the data is ready for analysis.

## Task 2: Summary statistics by group (15 points)

**The task:** a table with the mean and standard deviation of `Confidence_Score` and `Monthly_Income` for each group, plus a 3–4 sentence interpretation comparing the groups and stating the differences.

### How to solve it

1. `groupby('Group')` splits the data into Control and Treatment.
2. `.agg()` calculates the mean and standard deviation for each column within each group. That gives the 8 values the table needs.
3. **Calculate the differences yourself:** Treatment mean − Control mean, for both variables. Always state the order of subtraction.
4. **Interpret direction carefully.** For monthly income, higher is better.

**Remember:** these are *descriptive* statistics. They describe these 300 participants but don't yet tell you whether the gap would hold for everyone. That's what Tasks 3 and 4 test.

In [ ]:
summary_stats = (
    df.groupby("Group")
    .agg(
        {
            "Confidence_Score": ["mean", "std"],
            "Monthly_Income": ["mean", "std"],
        }
    )
    .round(2)
)
display(summary_stats)

means = df.groupby("Group")[
    ["Confidence_Score", "Monthly_Income"]
].mean()
diff = means.loc["Treatment"] - means.loc["Control"]
print(
    "Confidence_Score difference (Treatment - Control): "
    f"{diff['Confidence_Score']:.2f} points"
)
print(
    "Monthly_Income difference (Treatment - Control): "
    f"${diff['Monthly_Income']:,.2f}"
)
print(
    "Annual income difference: "
    f"${diff['Monthly_Income'] * 12:,.2f}"
)

### Model interpretation

1. Treatment participants are more confident: 7.49 vs. 6.00, **1.48 points higher**.
2. Treatment participants earn $3,569.09 a month vs. $2,851.66, **$717.43 more per month**.
3. Annually that is $717.43 × 12 = **$8,609.18** more.
4. The Control average ($2,851.66) is **below** the $3,200 self-sufficiency wage; the Treatment average ($3,569.09) is **above** it.
5. On descriptive statistics alone, the program appears to improve both confidence and income, enough to move the average participant past self-sufficiency. Tasks 3–5 test whether this holds beyond the sample.

## Task 3: Two-sample t-test on Confidence_Score (15 points)

**The task:** run an independent two-sample t-test, report t (4 decimals) and the p-value, and explain the hypothesis decision, the practical meaning, statistical significance, and the size of the difference.

### How to solve it

1. **Hypotheses.** H₀: the population mean confidence scores are equal. H₁: they differ (two-sided).
2. **Separate the two groups' scores**, then run `stats.ttest_ind(control, treatment)`.
3. **Decision rule:** if p < 0.05, reject H₀; otherwise fail to reject it. Never say "accept H₀."
4. **Check the variance assumption** with Levene's test, and compare with Welch's version.
5. **Report the size of the effect,** not only the p-value: the raw difference in points and Cohen's d (difference ÷ pooled SD; about 0.2 small, 0.5 medium, 0.8 large).
6. **Very small p-values** print as 0.0000; report them as p < 0.0001.

In [ ]:
control_s = df[df["Group"] == "Control"][
    "Confidence_Score"
]
treatment_s = df[df["Group"] == "Treatment"][
    "Confidence_Score"
]

t_stat, p_value = stats.ttest_ind(control_s, treatment_s)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4g}")
print(
    "Result:",
    (
        "SIGNIFICANT"
        if p_value < 0.05
        else "NOT SIGNIFICANT"
    ),
)

lev_p = stats.levene(control_s, treatment_s).pvalue
t_w, p_w = stats.ttest_ind(
    control_s, treatment_s, equal_var=False
)
print(
    f"\nLevene p = {lev_p:.4f}; Welch t = {t_w:.4f}, p = "
    f"{p_w:.4g}"
)
print(
    "Difference (Treatment - Control): "
    f"{treatment_s.mean() - control_s.mean():.2f} points"
)
print(
    f"Cohen's d: {cohens_d(treatment_s, control_s):.2f}"
)

### Model interpretation

The t-test gives **t = -7.6034, p < 0.0001**. Because p is far below 0.05, we **reject the null hypothesis** of equal average confidence. The t-statistic is negative only because Control was entered first; Treatment's mean is higher (7.49 vs. 6.00). A gap this large would be extremely unlikely if the 12-week Fast-Track training program had no effect on confidence, so it reflects a real difference, not chance. Practically, the gap is **1.48 points on the 1–10 confidence scale**, with Cohen's d = 0.88, a large effect. Levene's test (p = 0.0285) suggests unequal variances, but Welch's test gives the same conclusion.

## Task 4: 95% confidence intervals for Monthly_Income (15 points)

**The task:** report the lower bound, mean, and upper bound for each group, then explain what each interval means, whether they overlap, and the practical difference in dollars.

### How to solve it

1. **Formula:** CI = mean ± t* × SE, where SE = SD ÷ √n and t* = `t_dist.ppf(0.975, n − 1)` (about 1.98 for n = 150).
2. **Run the function once per group.**
3. **Meaning:** "We are 95% confident the true average monthly income for all participants like these lies between L and U." The interval describes the *average*, not individual participants.
4. **Overlap:** if the two intervals don't overlap, that is clear evidence of a real difference. Overlapping intervals are only a clue, not proof of no difference.
5. **For decisions,** a confidence interval for the *difference* is even more useful. It is included below as an extra.

In [ ]:
control_y = df[df["Group"] == "Control"]["Monthly_Income"]
treatment_y = df[df["Group"] == "Treatment"][
    "Monthly_Income"
]

mean_c, lower_c, upper_c = confidence_interval_95(
    control_y
)
mean_t, lower_t, upper_t = confidence_interval_95(
    treatment_y
)
print(
    f"Control:   mean ${mean_c:,.2f}, 95% CI "
    f"[${lower_c:,.2f}, ${upper_c:,.2f}]"
)
print(
    f"Treatment: mean ${mean_t:,.2f}, 95% CI "
    f"[${lower_t:,.2f}, ${upper_t:,.2f}]"
)
print(
    "Intervals overlap?",
    not (upper_c < lower_t or upper_t < lower_c),
)

# Extra: 95% CI for the difference (Welch)
diff = mean_t - mean_c
se_diff = np.sqrt(
    control_y.var(ddof=1) / len(control_y)
    + treatment_y.var(ddof=1) / len(treatment_y)
)
print(
    f"\nDifference (Treatment - Control): ${diff:,.2f}, "
    f"approx. 95% CI [${diff - 1.97 * se_diff:,.2f}, "
    f"${diff + 1.97 * se_diff:,.2f}]"
)

### Model interpretation

- **Control:** mean $2,851.66, 95% CI [$2,772.30, $2,931.02].
- **Treatment:** mean $3,569.09, 95% CI [$3,499.27, $3,638.92].
- **Overlap:** none. The Control interval's top is about $568.25 below the Treatment interval's bottom, strong evidence of a real income difference.
- **Self-sufficiency ($3,200):** the Control interval lies entirely *below* $3,200, so the typical Control participant is unlikely to reach self-sufficiency. The Treatment interval lies entirely *above* $3,200, so the typical trainee is likely to exceed it.
- **Estimated increase:** about **$717.43 per month** (95% CI for the difference roughly $612.16 to $822.71), or **$8,609.18 per year**.

## Task 5: Regression analysis (20 points)

**The task:** regress `Monthly_Income` on `Confidence_Score` and `Group_Treatment` (1 = Treatment, 0 = Control); report the full summary, R², both coefficients and p-values, and interpret them.

### How to solve it

1. **Create the dummy variable:** `Group_Treatment = 1` for Treatment, `0` for Control. Control becomes the reference group.
2. **Add a constant** (`sm.add_constant`) so the model has an intercept, then fit `sm.OLS(y, X)`.
3. **Read the output in this order:** R², then each coefficient (sign and size), then each p-value, then what the group effect means *after controlling for the score*.
   - **Confidence_Score coefficient:** the change in monthly income for a 1-point higher score, *holding group constant*.
   - **Group_Treatment coefficient:** the difference between Treatment and Control *for participants with the same score*.
   - **R²:** the share of the variation in monthly income the model explains (not "accuracy").
4. **Signs matter:** here a positive coefficient is good news (higher monthly income).
5. **Why both effects matter:** if the group effect stays significant after controlling for the score, the program adds value *beyond* making people more confident.

In [ ]:
df["Group_Treatment"] = (
    df["Group"] == "Treatment"
).astype(int)
X = sm.add_constant(
    df[["Confidence_Score", "Group_Treatment"]]
)
y = df["Monthly_Income"]
model = sm.OLS(y, X).fit()
print(model.summary())

print(f"\nR-squared = {model.rsquared:.4f}")
for v in ["Confidence_Score", "Group_Treatment"]:
    print(
        f"{v}: beta = {model.params[v]:,.2f}, p = "
        f"{model.pvalues[v]:.4g}"
    )

### Model interpretation

- **Model fit:** R² = **0.6287**, so the model explains about **63%** of the variation in monthly income: a strong fit by the course guideline (above 50%).
- **Confidence_Score:** β = **$174.42**, p < 0.0001. "For every 1-point increase in confidence score, monthly income increases by about $174.42." This is statistically significant.
- **Group_Treatment:** β = **$458.71**, p < 0.0001. "Participants in the vocational training program earn about $458.71 more per month than those in standard assistance, even after accounting for differences in confidence levels." This is statistically significant.
- **Why controlling for confidence matters:** it separates the program's direct effect (skills, certification, placement) from its effect through confidence, so the council knows the benefit isn't just a morale boost.
- **How the program creates value:** both coefficients are significant, so it works through two paths: the confidence gain (1.48 points × $174.42 ≈ $258.72) plus a direct effect of about $458.71. Together these account for the $717.43 total gap.
- **Conclusion:** yes, the program has a real, independent effect on income in this pilot.

## Task 6: Visualizations (15 points)

**The task:** an overlapping histogram and side-by-side box plots of `Confidence_Score`, each with labels, a title, and a legend where needed, plus 2–3 sentences interpreting each.

### How to solve it

- **Histogram:** plot both groups on the same axes with transparency (`alpha=0.6`) so the overlap shows. Use the same bins for both (10 bins across 1–10). Look at *where each group's bars pile up* and *which group has more high and low scores*.
- **Box plot:** the box spans Q1 to Q3 (the middle half), the line is the median, the whiskers reach the furthest point within 1.5 × IQR, and dots are outliers. Compare medians, box widths (consistency), and outliers.
- **Template fix:** in the assignment's code, `df.boxplot(by='Group')` opens a new figure and leaves the second subplot empty. Pass `ax=` to keep both charts side by side, as below.
- **Connect it to the decision:** say what the pattern means for the program, not only what the chart shows.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7, 9))
bins = np.linspace(1, 10, 11)
ax[0].hist(
    control_s,
    bins=bins,
    alpha=0.6,
    label="Control",
    color="#3D5A80",
    edgecolor="white",
)
ax[0].hist(
    treatment_s,
    bins=bins,
    alpha=0.6,
    label="Treatment",
    color="#E9A23B",
    edgecolor="white",
)
ax[0].set_xlabel("Confidence Score")
ax[0].set_ylabel("Frequency")
ax[0].set_title("Distribution of Confidence Score")
ax[0].legend()

df.boxplot(
    column="Confidence_Score", by="Group", ax=ax[1]
)
ax[1].set_xlabel("Group")
ax[1].set_ylabel("Confidence Score")
ax[1].set_title("Confidence Score by Group")
plt.suptitle("")
plt.tight_layout()
plt.show()

for g, sc in [
    ("Control", control_s),
    ("Treatment", treatment_s),
]:
    q1, q3 = sc.quantile([0.25, 0.75])
    print(
        f"{g}: median {sc.median():.2f}, IQR "
        f"{q3 - q1:.2f}, share scoring 8+: "
        f"{(sc >= 8).mean():.1%}"
    )

### Model interpretation

**Histogram.** Treatment scores are concentrated toward the top of the scale: 43% of Treatment participants scored 8 or higher, versus 19% of Control. Control scores are more spread out, with more low scores (30% scored 5 or below, versus 9.3% for Treatment). The two distributions overlap, so not every Treatment participant felt more confident than every Control participant, but the whole Treatment distribution has shifted to the right.

**Box plot.** Treatment's median (7.55) is well above Control's (5.90). Treatment's box is narrower (IQR 2.07 vs. 2.85), so its scores are more consistent and predictable. Control has no outliers; Treatment has one low outlier at 3.1, likely individuals who didn't engage with the program or had an unusually poor experience. These are worth following up rather than deleting. Together, the charts support the t-test: Treatment participants are consistently more confident.

## Task 7: Policy recommendation memo (10 points)

**The task:** a 1–2 page memo in Markdown cells with a clear YES or NO, key findings with specific numbers, a cost-benefit analysis, a recommendation, and limitations.

### How to solve it

1. **Economic analysis.** Monthly increase = Task 2 difference; annual increase = monthly × 12; payback months = $5,000 ÷ monthly increase; annual ROI = annual increase ÷ $5,000 × 100.
2. **Self-sufficiency analysis.** Estimate the share of each group earning above $3,200. You can count it directly from the data, or estimate it from the mean and SD with a normal distribution; both are shown below.
3. **Policy decision:** a clear yes or no, the primary reason, two or three limitations, and the extra data that would strengthen the case.

In [ ]:
gain_m = treatment_y.mean() - control_y.mean()
cost = 5000
print(
    f"Monthly increase: ${gain_m:,.2f};  annual increase: "
    f"${12 * gain_m:,.2f}"
)
print(
    f"Payback: {cost / gain_m:.1f} months;  annual ROI: "
    f"{12 * gain_m / cost * 100:.1f}%"
)
for g, inc in [
    ("Control", control_y),
    ("Treatment", treatment_y),
]:
    counted = (inc > 3200).mean()
    normal_est = 1 - stats.norm.cdf(
        3200, inc.mean(), inc.std()
    )
    print(
        f"{g}: {counted:.1%} earn above $3,200 (counted); "
        f"{normal_est:.1%} (normal estimate)"
    )
extra = (
    (treatment_y > 3200).mean()
    - (control_y > 3200).mean()
) * 150
print(
    "Additional participants reaching self-sufficiency "
    f"per 150 enrolled: about {extra:.0f}"
)

### Model memo

**TO:** City Council  **FROM:** Department of Labor Analyst  **DATE:** [date]  **SUBJECT:** Funding the Fast-Track Vocational Training Program

**Economic analysis.**
1. Average monthly income increase: **$717.43**.
2. Average annual increase: **$8,609.18**.
3. Payback on $5,000 per participant: $5,000 ÷ $717.43 ≈ **7.0 months**.
4. Annual ROI: $8,609.18 ÷ $5,000 × 100 ≈ **172%**.

**Self-sufficiency analysis.**
1. About **25%** of Control participants earn above $3,200 (25.3% counted; about 24% estimated from the mean and SD).
2. About **82%** of Treatment participants earn above $3,200 (82.0% counted; about 80% estimated).
3. That is roughly 57 percentage points more, or about **85 more of every 150 participants** reaching self-sufficiency.

**Policy decision.**
1. **YES. The council should increase funding for Fast-Track.**
2. **Primary reason:** each $5,000 investment is recovered in participants' earnings in about seven months, and the program moves roughly 85 more of every 150 participants above the self-sufficiency wage. The effect remains significant after controlling for confidence (β = $458.71, p < 0.0001).
3. **Limitations:** (a) earnings were measured over one 12-month pilot, so long-term persistence is unknown; (b) if participants weren't randomly assigned, more motivated residents may have chosen the training (selection bias); (c) a strong local job market during the pilot may have helped (confounding), and the gains may shrink if the program scales faster than employer partnerships.
4. **Additional data:** two- to three-year earnings follow-up, job retention, reduced public-assistance spending (fiscal savings to the city), and results by industry track (healthcare, IT, skilled trades).

---
## What separates a full-credit answer

Across every task, the strongest submissions do five things:

1. **State the number**: means, difference, t, p, CI, coefficient, or R².
2. **Make the decision**: reject or fail to reject H₀, with α = 0.05.
3. **Translate it into the scenario**: customers, patients, or residents, not "the variable."
4. **Give the practical size**: points, dollars, Cohen's d, payback, or ROI.
5. **Name a limitation**: sample, design, persistence, or confounding.

Number → decision → plain English → practical meaning → limitation.